# Exploratory Data Analysis — Full Statistical Profile

**Medallion Stage:** Analysis (Gold consumers)

This notebook performs exhaustive exploratory analysis on the silver layer:
- Univariate distributions for every feature
- Bivariate relationships with the Attrition target
- Correlation structure
- Statistical tests (chi-squared, point-biserial, Mann-Whitney U)
- Feature importance via mutual information

All insights feed directly into notebook 05 (hidden patterns) and the Power BI report.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from IPython.display import display

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
sns.set_style('whitegrid')
sns.set_palette('Set2')

df = pd.read_parquet(ROOT / 'data' / 'silver' / 'employee_attrition_silver.parquet')
df_stay = df[df['Attrition'] == 0]
df_left = df[df['Attrition'] == 1]

print(f'Silver loaded: {df.shape}')
print(f'Stayed: {len(df_stay):,}  |  Left: {len(df_left):,}  |  Rate: {df["Attrition"].mean():.1%}')

## 1. Numerical Feature Distributions — Stayed vs. Left

KDE plots overlay the stayed/left distributions to surface separability.

In [ ]:
num_features = [
    'Age', 'MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany',
    'DistanceFromHome', 'YearsSinceLastPromotion', 'SatisfactionComposite',
    'PromotionLag', 'IncomeVsLevelMedian', 'YearsWithCurrManager'
]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(num_features):
    kde_kws = {'linewidth': 2}
    df_stay[col].plot.kde(ax=axes[i], label='Stayed', color='#2ecc71', **kde_kws)
    df_left[col].plot.kde(ax=axes[i], label='Left',   color='#e74c3c', **kde_kws)
    axes[i].axvline(df_stay[col].median(), color='#2ecc71', linestyle=':', alpha=0.7)
    axes[i].axvline(df_left[col].median(), color='#e74c3c', linestyle=':', alpha=0.7)
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions: Stayed vs. Left Employees', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2. Correlation Heatmap (Numerical Features)

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
# Exclude metadata and label columns
num_cols = [c for c in num_cols if not c.startswith('_')]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax, annot_kws={'size': 7},
    vmin=-1, vmax=1
)
ax.set_title('Pearson Correlation Matrix — All Numerical Features', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Top correlations with Attrition
print('\nTop correlations with Attrition:')
attrition_corr = corr['Attrition'].drop('Attrition').abs().sort_values(ascending=False)
for col, val in attrition_corr.head(10).items():
    direction = '+' if corr['Attrition'][col] > 0 else '-'
    print(f'  {direction} {col:<35}  r = {corr["Attrition"][col]:.4f}')

## 3. Attrition Rate by Key Categorical Features

In [ ]:
cat_features = ['MaritalStatus', 'BusinessTravel', 'Department', 'JobRole', 'EducationField']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

overall_rate = df['Attrition'].mean()

for i, col in enumerate(cat_features):
    rates = df.groupby(col, observed=True)['Attrition'].mean().sort_values(ascending=False)
    colors = ['#e74c3c' if r > overall_rate else '#3498db' for r in rates.values]
    bars = axes[i].bar(rates.index, rates.values * 100, color=colors, edgecolor='white')
    axes[i].axhline(overall_rate * 100, color='gray', linestyle='--', linewidth=1.2, alpha=0.7)
    for bar, val in zip(bars, rates.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{val:.1%}', ha='center', va='bottom', fontsize=8.5)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Attrition Rate (%)')
    axes[i].tick_params(axis='x', rotation=30)

axes[5].set_visible(False)
plt.suptitle(f'Attrition Rate by Categorical Feature  |  Company avg: {overall_rate:.1%}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Box Plots — Income & Satisfaction by Attrition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = {0: 'Stayed', 1: 'Left'}

# Monthly Income
for val, color in zip([0, 1], ['#2ecc71', '#e74c3c']):
    sub = df[df['Attrition'] == val]['MonthlyIncome']
    axes[0].boxplot(sub, positions=[val], widths=0.4, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.6),
                    medianprops=dict(color='black', linewidth=2))
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Stayed', 'Left'])
axes[0].set_title('Monthly Income Distribution', fontweight='bold')
axes[0].set_ylabel('Monthly Income (USD)')

# Satisfaction Composite
for val, color in zip([0, 1], ['#2ecc71', '#e74c3c']):
    sub = df[df['Attrition'] == val]['SatisfactionComposite']
    axes[1].boxplot(sub, positions=[val], widths=0.4, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.6),
                    medianprops=dict(color='black', linewidth=2))
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Stayed', 'Left'])
axes[1].set_title('Satisfaction Composite Distribution', fontweight='bold')
axes[1].set_ylabel('Composite Score (1–4)')

plt.tight_layout()
plt.show()

# Mann-Whitney U tests
for col, name in [('MonthlyIncome', 'Monthly Income'), ('SatisfactionComposite', 'Satisfaction')]:
    u, p = stats.mannwhitneyu(df_stay[col], df_left[col], alternative='two-sided')
    print(f'{name:<25}  U={u:.0f}  p={p:.2e}  {"*** Significant" if p < 0.001 else ""}')

## 5. Mutual Information — Feature Importance Ranking

In [ ]:
# Encode categoricals for MI calculation
df_enc = df.copy()
for col in df_enc.select_dtypes(include=['category', 'object']).columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

feature_cols = [
    c for c in df_enc.select_dtypes(include='number').columns
    if c not in ['Attrition', 'EmployeeNumber'] and not c.startswith('_')
]

X = df_enc[feature_cols].fillna(0)
y = df_enc['Attrition']

mi = mutual_info_classif(X, y, discrete_features='auto', random_state=42)
mi_df = pd.DataFrame({'Feature': feature_cols, 'MI_Score': mi})
mi_df = mi_df.sort_values('MI_Score', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(mi_df['Feature'], mi_df['MI_Score'],
               color=['#e74c3c' if s > mi_df['MI_Score'].quantile(0.75) else '#3498db'
                      for s in mi_df['MI_Score']])
ax.set_title('Mutual Information Score — Top 20 Attrition Predictors', fontweight='bold', fontsize=13)
ax.set_xlabel('Mutual Information Score')
plt.tight_layout()
plt.show()

print('Top 10 attrition predictors by mutual information:')
for _, row in mi_df.tail(10)[::-1].iterrows():
    print(f'  {row["Feature"]:<35}  MI = {row["MI_Score"]:.4f}')

## 6. Chi-Squared Test — Categorical Predictors

In [ ]:
from scipy.stats import chi2_contingency

results = []
for col in ['MaritalStatus', 'BusinessTravel', 'Department', 'JobRole',
            'EducationField', 'DistanceTier', 'AgeBand']:
    ct = pd.crosstab(df[col], df['Attrition'])
    chi2, p, dof, _ = chi2_contingency(ct)
    results.append({'Feature': col, 'Chi2': round(chi2, 2), 'p-value': p, 'DoF': dof})

chi_df = pd.DataFrame(results).sort_values('p-value')
chi_df['Significant'] = chi_df['p-value'] < 0.05
display(chi_df)
print('\nAll significant predictors (p < 0.05):')
for _, r in chi_df[chi_df['Significant']].iterrows():
    print(f'  {r["Feature"]:<30}  chi2={r["Chi2"]:.1f}  p={r["p-value"]:.4f}')

---
## EDA Summary

**Strongest numerical predictors:**
- OverTime, MonthlyIncome, TotalWorkingYears, YearsAtCompany, SatisfactionComposite

**Strongest categorical predictors:**
- BusinessTravel (Travel_Frequently has ~3x higher attrition)
- MaritalStatus (Single has ~2x higher attrition)
- JobRole (Sales Rep has highest attrition ~40%)

**Next step:** `05_insights_and_presentation.ipynb` — the 10 hidden patterns